<a href="https://colab.research.google.com/github/Umairfarooq7893/Global-Burden-of-Disease-Regional-Demographic-Analysis-2000-2023-/blob/main/1601_25_737_197%2C601%2C919.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

ihme_files = [
    "/content/sample_data/IHME-GBD_2023_DATA-7985af47-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-0a9f0f3f-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-0cb8ed3e-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-f4648327-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-324fa6d6-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-f5df3c4f-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-8bfd437b-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-fbd7a556-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-88030a9d-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-f07d3e31-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-35e5eddc-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-854684ae-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-87cf5a11-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-5f485e02-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-0043ea65-1.csv",
    "/content/sample_data/IHME-GBD_2023_DATA-678efd4d-1.csv",
]

print("Total files:", len(ihme_files))

df = pd.read_csv(ihme_files[0])

print("Shape:", df.shape)
print()
print("Columns:")
for c in df.columns:
    print("  -", c)
print()
print("Data types:")
print(df.dtypes)


text_cols = ["location_name", "sex_name", "age_name", "cause_name",
             "measure_name", "metric_name", "population_group_name"]

for col in text_cols:
    values = sorted(df[col].dropna().unique().tolist())
    print(col, "->", len(values), "unique values")
    print("   ", values)
    print()


print("Year range :", df["year"].min(), "to", df["year"].max())
print("Years      :", sorted(df["year"].unique().tolist()))
print()
print("Value column stats:")
print(df[["val", "upper", "lower"]].describe().round(2))
print()
print("Missing values per column:")
print(df.isna().sum())

Total files: 16
Shape: (9288, 18)

Columns:
  - population_group_id
  - population_group_name
  - measure_id
  - measure_name
  - location_id
  - location_name
  - sex_id
  - sex_name
  - age_id
  - age_name
  - cause_id
  - cause_name
  - metric_id
  - metric_name
  - year
  - val
  - upper
  - lower

Data types:
population_group_id        int64
population_group_name     object
measure_id                 int64
measure_name              object
location_id                int64
location_name             object
sex_id                     int64
sex_name                  object
age_id                     int64
age_name                  object
cause_id                   int64
cause_name                object
metric_id                  int64
metric_name               object
year                       int64
val                      float64
upper                    float64
lower                    float64
dtype: object
location_name -> 1 unique values
    ['Oceania']

sex_name -> 2 unique value

In [ ]:
parts = []
for f in ihme_files:
    temp = pd.read_csv(f)
    temp["source_file"] = f.split("/")[-1]
    parts.append(temp)

ihme = pd.concat(parts, ignore_index=True)

print("Combined shape:", ihme.shape)
print()
print("Regions in combined table:")
print(sorted(ihme["location_name"].unique().tolist()))



print("Rows per region:")
print(ihme["location_name"].value_counts().sort_index())
print()
print("Causes in combined table:")
print(sorted(ihme["cause_name"].unique().tolist()))
print()
print("Sexes:", sorted(ihme["sex_name"].unique().tolist()))
print()
print("Number of age bands:", ihme["age_name"].nunique())
print("Age bands:", sorted(ihme["age_name"].unique().tolist(), key=lambda x: str(x)))

Combined shape: (148608, 19)

Regions in combined table:
['Australasia', 'Central Asia', 'Central Europe', 'Central Sub-Saharan Africa', 'China', 'Eastern Europe', 'Eastern Sub-Saharan Africa', 'High-income North America', 'India', 'Latin America and Caribbean', 'North Africa and Middle East', 'Oceania', 'Southeast Asia', 'Southern Sub-Saharan Africa', 'Western Europe', 'Western Sub-Saharan Africa']
Rows per region:
location_name
Australasia                     9288
Central Asia                    9288
Central Europe                  9288
Central Sub-Saharan Africa      9288
China                           9288
Eastern Europe                  9288
Eastern Sub-Saharan Africa      9288
High-income North America       9288
India                           9288
Latin America and Caribbean     9288
North Africa and Middle East    9288
Oceania                         9288
Southeast Asia                  9288
Southern Sub-Saharan Africa     9288
Western Europe                  9288
Western Sub

In [ ]:
dead_cols = [
    "population_group_id",
    "population_group_name",
    "measure_id",
    "measure_name",
    "metric_id",
    "metric_name",
]

base = ihme.drop(columns=dead_cols)
print("After dropping dead columns:", base.shape)
print(base.columns.tolist())


id_cols = ["location_id", "sex_id", "age_id", "cause_id"]
base = base.drop(columns=id_cols)
print("After dropping IDs:", base.shape)
print(base.columns.tolist())


base = base.rename(columns={
    "location_name": "region",
    "sex_name": "sex",
    "age_name": "age",
    "cause_name": "cause",
    "val": "deaths",
    "upper": "deaths_upper",
    "lower": "deaths_lower",
})
print(base.columns.tolist())
base.head(3)
base["region"] = base["region"].replace({
    "High-income North America": "North America"
})

print("Regions now:")
print(sorted(base["region"].unique().tolist()))


for col in ["region", "sex", "age", "cause"]:
    base[col] = base[col].astype(str).str.strip()

base["year"] = base["year"].astype("int64")
base["deaths"] = base["deaths"].astype("float64")
base["deaths_upper"] = base["deaths_upper"].astype("float64")
base["deaths_lower"] = base["deaths_lower"].astype("float64")

print("Done.")
print(base.dtypes)


key = ["region", "year", "cause", "sex", "age"]
dupes = base.duplicated(subset=key).sum()
print("Duplicate rows on the key:", dupes)



base["uncertainty"] = (base["deaths_upper"] - base["deaths_lower"]) / base["deaths"]

# handle division-by-zero (rare, but possible when deaths = 0)
base["uncertainty"] = base["uncertainty"].replace([np.inf, -np.inf], np.nan)

print(base["uncertainty"].describe().round(3))

print("Shape               :", base.shape)
print("Regions             :", base["region"].nunique())
print("Causes              :", base["cause"].nunique())
print("Sexes               :", base["sex"].nunique())
print("Age bands           :", base["age"].nunique())
print("Years               :", base["year"].min(), "to", base["year"].max())
print()
print("Any missing values  :", base.isna().sum().sum())
print("Any negative deaths :", (base["deaths"] < 0).sum())
print()
print()
print()
base.to_csv("/content/sample_data/base_clean.csv", index=False)
print("Saved base_clean.csv")

After dropping dead columns: (148608, 13)
['location_id', 'location_name', 'sex_id', 'sex_name', 'age_id', 'age_name', 'cause_id', 'cause_name', 'year', 'val', 'upper', 'lower', 'source_file']
After dropping IDs: (148608, 9)
['location_name', 'sex_name', 'age_name', 'cause_name', 'year', 'val', 'upper', 'lower', 'source_file']
['region', 'sex', 'age', 'cause', 'year', 'deaths', 'deaths_upper', 'deaths_lower', 'source_file']
Regions now:
['Australasia', 'Central Asia', 'Central Europe', 'Central Sub-Saharan Africa', 'China', 'Eastern Europe', 'Eastern Sub-Saharan Africa', 'India', 'Latin America and Caribbean', 'North Africa and Middle East', 'North America', 'Oceania', 'Southeast Asia', 'Southern Sub-Saharan Africa', 'Western Europe', 'Western Sub-Saharan Africa']
Done.
region           object
sex              object
age              object
cause            object
year              int64
deaths          float64
deaths_upper    float64
deaths_lower    float64
source_file      object
dty

In [ ]:
#SECTION 3
region_year_total = (
    base.groupby(["region", "year"], as_index=False)["deaths"]
    .sum()
    .rename(columns={"deaths": "total_deaths"})
)

print("Shape:", region_year_total.shape)
print(region_year_total.head())

check_2023 = region_year_total[region_year_total["year"] == 2023]
check_2023 = check_2023.sort_values("total_deaths", ascending=False)
print(check_2023.to_string(index=False))


region_cause_year = (
    base.groupby(["region", "year", "cause"], as_index=False)
    .agg(
        deaths=("deaths", "sum"),
        deaths_upper=("deaths_upper", "sum"),
        deaths_lower=("deaths_lower", "sum"),
    )
)

print("Shape:", region_cause_year.shape)
print(region_cause_year.head(10))



test = (
    region_cause_year.groupby(["region", "year"], as_index=False)["deaths"]
    .sum()
    .rename(columns={"deaths": "cause_sum"})
)

compare = test.merge(region_year_total, on=["region", "year"])
compare["diff"] = (compare["cause_sum"] - compare["total_deaths"]).abs()

print("Max difference:", compare["diff"].max())
print("Rows with diff > 0.01:", (compare["diff"] > 0.01).sum())



region_cause_year = region_cause_year.merge(
    region_year_total, on=["region", "year"], how="left"
)

region_cause_year["share"] = (
    region_cause_year["deaths"] / region_cause_year["total_deaths"]
) * 100

# Round for readability, keep raw values too
region_cause_year["share"] = region_cause_year["share"].round(3)

print(region_cause_year.head(10))


region_cause_year["uncertainty"] = (
    (region_cause_year["deaths_upper"] - region_cause_year["deaths_lower"])
    / region_cause_year["deaths"]
).round(3)

print(region_cause_year[["region", "year", "cause", "deaths", "share", "uncertainty"]].head())


region_cause_year.to_csv("/content/sample_data/region_cause_year.csv", index=False)
region_year_total.to_csv("/content/sample_data/region_year_total.csv", index=False)
print("Saved both.")

Shape: (384, 3)
        region  year  total_deaths
0  Australasia  2000  58224.359219
1  Australasia  2001  59022.759133
2  Australasia  2002  60240.555171
3  Australasia  2003  60773.944553
4  Australasia  2004  61308.938298
                      region  year  total_deaths
                       India  2023  3.893171e+06
                       China  2023  3.652851e+06
  Western Sub-Saharan Africa  2023  2.784228e+06
  Eastern Sub-Saharan Africa  2023  2.127165e+06
              Southeast Asia  2023  1.905770e+06
              Western Europe  2023  1.738490e+06
 Latin America and Caribbean  2023  1.571104e+06
North Africa and Middle East  2023  1.191890e+06
               North America  2023  1.173975e+06
  Central Sub-Saharan Africa  2023  8.350821e+05
              Eastern Europe  2023  7.462755e+05
 Southern Sub-Saharan Africa  2023  4.855162e+05
              Central Europe  2023  4.636762e+05
                Central Asia  2023  1.861582e+05
                 Australasia  2023  8.4

In [ ]:
#SECTION 4

latest = region_cause_year[region_cause_year["year"] == 2023]

pivot_2023 = latest.pivot_table(
    index="region",
    columns="cause",
    values="share",
    aggfunc="sum"
).round(2)

# Make it readable — sort columns by global share
order = latest.groupby("cause")["deaths"].sum().sort_values(ascending=False).index.tolist()
pivot_2023 = pivot_2023[order]

pivot_2023





rank_2023 = latest.pivot_table(
    index="region",
    columns="cause",
    values="share",
    aggfunc="sum"
)

# Rank each region's causes from largest to smallest
ranks = rank_2023.rank(axis=1, ascending=False).astype("Int64")

print("Rank of each cause within each region, 2023 (1 = leading cause):")
ranks

leading = rank_2023.idxmax(axis=1).rename("leading_cause")
leading_share = rank_2023.max(axis=1).round(2).rename("leading_share")

summary = pd.concat([leading, leading_share], axis=1)
print(summary.to_string())


infectious = [
    "HIV/AIDS and sexually transmitted infections",
    "Respiratory infections and tuberculosis",
    "Enteric infections",
    "Other infectious diseases",
    "Neglected tropical diseases and malaria",
    "Nutritional deficiencies",
    "Maternal and neonatal disorders",
]

non_infectious = [
    "Total cancers",
    "Injuries",
]

# sanity check: the two lists should cover all 9 causes
all_causes = set(infectious + non_infectious)
print("Covered causes:", len(all_causes))
print("Missing from lists:", set(base["cause"].unique()) - all_causes)


grouped = latest.copy()
grouped["category"] = grouped["cause"].apply(
    lambda c: "Infectious + maternal + nutritional" if c in infectious else "Non-infectious"
)

category_share = (
    grouped.groupby(["region", "category"])["share"]
    .sum()
    .unstack()
    .round(2)
)

print("Share of deaths by broad category, 2023:")
category_share

Rank of each cause within each region, 2023 (1 = leading cause):
                                                             leading_cause  leading_share
region                                                                                   
Australasia                                                  Total cancers          77.45
Central Asia                                                 Total cancers          42.29
Central Europe                                               Total cancers          70.84
Central Sub-Saharan Africa                                        Injuries          22.47
China                                                        Total cancers          69.46
Eastern Europe                                               Total cancers          56.53
Eastern Sub-Saharan Africa                                        Injuries          17.98
India                                                        Total cancers          26.88
Latin America and Caribbean        

category,Infectious + maternal + nutritional,Non-infectious
region,,
Australasia,7.03,92.97
Central Asia,36.62,63.38
Central Europe,15.75,84.25
Central Sub-Saharan Africa,65.74,34.26
China,16.18,83.82
Eastern Europe,13.00,87.00
Eastern Sub-Saharan Africa,67.43,32.58
India,48.76,51.24
Latin America and Caribbean,30.07,69.93


In [ ]:
#SECTION 5

y2000 = region_cause_year[region_cause_year["year"] == 2000][["region", "cause", "share"]]
y2023 = region_cause_year[region_cause_year["year"] == 2023][["region", "cause", "share"]]

trend = y2000.merge(
    y2023,
    on=["region", "cause"],
    how="outer",
    suffixes=("_2000", "_2023"),
)

trend["share_2000"] = trend["share_2000"].fillna(0)
trend["share_2023"] = trend["share_2023"].fillna(0)
trend["change_pp"] = (trend["share_2023"] - trend["share_2000"]).round(2)

print("Trend table shape:", trend.shape)
trend.head(10)

print()
print()
print()

global_change = (
    trend.groupby("cause")[["share_2000", "share_2023", "change_pp"]]
    .mean()
    .round(2)
    .sort_values("change_pp", ascending=False)
)

# Cell 29 — print explicitly instead of relying on display
print(global_change.to_string())

print()
print()
print()

def top_risers(group, n=3):
    return group.nlargest(n, "change_pp")[["cause", "change_pp"]].to_dict("records")

def top_fallers(group, n=3):
    return group.nsmallest(n, "change_pp")[["cause", "change_pp"]].to_dict("records")

risers = trend.groupby("region").apply(top_risers).rename("top_risers")
fallers = trend.groupby("region").apply(top_fallers).rename("top_fallers")

for region in sorted(trend["region"].unique()):
    print(f"--- {region} ---")
    print("  Rising :", risers[region])
    print("  Falling:", fallers[region])
    print()

print()
print()
print()


hiv_southern = region_cause_year[
    (region_cause_year["region"] == "Southern Sub-Saharan Africa") &
    (region_cause_year["cause"] == "HIV/AIDS and sexually transmitted infections")
].sort_values("year")

hiv_southern[["year", "deaths", "share"]].round(2)

print()
print()
print()

india_china = region_cause_year[region_cause_year["region"].isin(["India", "China"])]

india_china_pivot = india_china.pivot_table(
    index=["region", "year"],
    columns="cause",
    values="share"
).round(2)

# Show only decade boundaries to keep it readable
india_china_pivot.loc[(slice(None), [2000, 2005, 2010, 2015, 2020, 2023]), :]

print()
print()
print()


resp = region_cause_year[
    region_cause_year["cause"] == "Respiratory infections and tuberculosis"
].copy()

resp_pivot = resp.pivot_table(
    index="region",
    columns="year",
    values="share"
)

resp_selected = resp_pivot[[2017, 2018, 2019, 2020, 2021, 2022, 2023]].round(2)
resp_selected["change_19_to_21"] = (resp_selected[2021] - resp_selected[2019]).round(2)

# Cell 33 — same
print(resp_selected.sort_values("change_19_to_21", ascending=False).to_string())


print()
print()
print()


infectious = [
    "HIV/AIDS and sexually transmitted infections",
    "Respiratory infections and tuberculosis",
    "Enteric infections",
    "Other infectious diseases",
    "Neglected tropical diseases and malaria",
    "Nutritional deficiencies",
    "Maternal and neonatal disorders",
]

region_cause_year["category"] = region_cause_year["cause"].apply(
    lambda c: "infectious" if c in infectious else "non_infectious"
)

category_totals = (
    region_cause_year.groupby(["region", "year", "category"])["deaths"]
    .sum()
    .unstack()
    .fillna(0)
)

category_totals["ratio"] = (
    category_totals["non_infectious"] / category_totals["infectious"]
).round(2)

# Show ratio in 2000 and 2023 side by side
ratio_2000 = category_totals.xs(2000, level="year")["ratio"].rename("ratio_2000")
ratio_2023 = category_totals.xs(2023, level="year")["ratio"].rename("ratio_2023")

transition = pd.concat([ratio_2000, ratio_2023], axis=1)
transition["shift"] = (transition["ratio_2023"] - transition["ratio_2000"]).round(2)
transition = transition.sort_values("ratio_2023", ascending=False)

# Cell 34 — same
print(transition.to_string())


print()
print()
print()


trend.to_csv("/content/sample_data/trend_2000_to_2023.csv", index=False)
category_totals.to_csv("/content/sample_data/category_totals_by_region_year.csv")
print("Saved.")

Trend table shape: (144, 5)



                                              share_2000  share_2023  change_pp
cause                                                                          
Total cancers                                      33.77       42.29       8.52
Injuries                                           18.83       19.96       1.13
Respiratory infections and tuberculosis            15.28       16.11       0.83
Nutritional deficiencies                            1.23        0.99      -0.24
Neglected tropical diseases and malaria             3.55        2.89      -0.66
Maternal and neonatal disorders                     8.49        6.45      -2.04
Other infectious diseases                           5.24        3.08      -2.16
HIV/AIDS and sexually transmitted infections        7.15        4.73      -2.41
Enteric infections                                  6.47        3.50      -2.97



--- Australasia ---
  Rising : [{'cause': 'Total cancers', 'change_pp': 0.49}, {'cause

/tmp/ipykernel_3216/1799624409.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  risers = trend.groupby("region").apply(top_risers).rename("top_risers")
/tmp/ipykernel_3216/1799624409.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fallers = trend.groupby("region").apply(top_fallers).rename("top_fallers")


In [ ]:
#SECTION 6

region_cause_year_sex = (
    base.groupby(["region", "year", "cause", "sex"], as_index=False)
    .agg(
        deaths=("deaths", "sum"),
        deaths_upper=("deaths_upper", "sum"),
        deaths_lower=("deaths_lower", "sum"),
    )
)

print("Shape:", region_cause_year_sex.shape)
region_cause_year_sex.head()

print()
print()
print()

summed = (
    region_cause_year_sex.groupby(["region", "year", "cause"], as_index=False)["deaths"]
    .sum()
    .rename(columns={"deaths": "deaths_from_sex"})
)

check = region_cause_year[["region", "year", "cause", "deaths"]].merge(
    summed, on=["region", "year", "cause"], how="outer"
)
check["diff"] = (check["deaths"] - check["deaths_from_sex"]).abs()

print("Max difference:", check["diff"].max())
print("Rows with diff > 0.01:", (check["diff"] > 0.01).sum())

print()
print()
print()

sex_by_cause = (
    region_cause_year_sex.groupby(["cause", "sex"], as_index=False)["deaths"]
    .sum()
    .pivot(index="cause", columns="sex", values="deaths")
)

sex_by_cause["total"] = sex_by_cause["Male"] + sex_by_cause["Female"]
sex_by_cause["male_share_%"] = (sex_by_cause["Male"] / sex_by_cause["total"] * 100).round(2)
sex_by_cause["ratio_M_F"] = (sex_by_cause["Male"] / sex_by_cause["Female"]).round(2)

sex_by_cause = sex_by_cause.sort_values("male_share_%", ascending=False)

print("Global sex breakdown by cause, 2000–2023:")
print(sex_by_cause.to_string())

print()
print()
print()

injuries = region_cause_year_sex[region_cause_year_sex["cause"] == "Injuries"]

injuries_pivot = (
    injuries.groupby(["region", "sex"], as_index=False)["deaths"]
    .sum()
    .pivot(index="region", columns="sex", values="deaths")
)

injuries_pivot["male_share_%"] = (
    injuries_pivot["Male"] / (injuries_pivot["Male"] + injuries_pivot["Female"]) * 100
).round(2)
injuries_pivot["ratio_M_F"] = (injuries_pivot["Male"] / injuries_pivot["Female"]).round(2)

injuries_pivot = injuries_pivot.sort_values("ratio_M_F", ascending=False)

print("Injuries deaths — male share and M/F ratio by region (2000–2023):")
print(injuries_pivot[["male_share_%", "ratio_M_F"]].to_string())

print()
print()
print()

hiv = region_cause_year_sex[
    region_cause_year_sex["cause"] == "HIV/AIDS and sexually transmitted infections"
]

hiv_pivot = (
    hiv.groupby(["region", "sex"], as_index=False)["deaths"]
    .sum()
    .pivot(index="region", columns="sex", values="deaths")
)

hiv_pivot["male_share_%"] = (
    hiv_pivot["Male"] / (hiv_pivot["Male"] + hiv_pivot["Female"]) * 100
).round(2)
hiv_pivot["ratio_M_F"] = (hiv_pivot["Male"] / hiv_pivot["Female"]).round(2)

hiv_pivot = hiv_pivot.sort_values("ratio_M_F", ascending=False)

print("HIV/AIDS deaths — male share and M/F ratio by region (2000–2023):")
print(hiv_pivot[["male_share_%", "ratio_M_F"]].to_string())

print()
print()
print()


mn = region_cause_year_sex[
    region_cause_year_sex["cause"] == "Maternal and neonatal disorders"
]

mn_pivot = (
    mn.groupby(["region", "sex"], as_index=False)["deaths"]
    .sum()
    .pivot(index="region", columns="sex", values="deaths")
)

mn_pivot["male_share_%"] = (
    mn_pivot["Male"] / (mn_pivot["Male"] + mn_pivot["Female"]) * 100
).round(2)

mn_pivot = mn_pivot.sort_values("male_share_%", ascending=False)

print("Maternal + neonatal deaths — male share by region:")
print(mn_pivot[["male_share_%"]].to_string())

print()
print()
print()

region_cause_year_sex.to_csv("/content/sample_data/region_cause_year_sex.csv", index=False)
sex_by_cause.to_csv("/content/sample_data/global_sex_by_cause.csv")
print("Saved.")

Shape: (6912, 7)



Max difference: 4.656612873077393e-10
Rows with diff > 0.01: 0



Global sex breakdown by cause, 2000–2023:
sex                                                 Female          Male         total  male_share_%  ratio_M_F
cause                                                                                                          
Injuries                                      3.029920e+07  7.009809e+07  1.003973e+08         69.82       2.31
Total cancers                                 7.640118e+07  1.000693e+08  1.764705e+08         56.71       1.31
Respiratory infections and tuberculosis       4.248275e+07  5.557913e+07  9.806188e+07         56.68       1.31
Neglected tropical diseases and malaria       1.040719e+07  1.135219e+07  2.175938e+07         52.17       1.09
Other infectious diseases                     1.252680e+07  1.313087e+07  2.565767e+07         51.18       1.05
Maternal and neonatal disorders               2.463829e+07  2.517691e+07  4.981520e+07  

In [ ]:
#SECTION 7

region_cause_year_age = (
    base.groupby(["region", "year", "cause", "age"], as_index=False)
    .agg(
        deaths=("deaths", "sum"),
        deaths_upper=("deaths_upper", "sum"),
        deaths_lower=("deaths_lower", "sum"),
    )
)

print("Shape:", region_cause_year_age.shape)
print(region_cause_year_age.head(3).to_string())

print()
print()
print()

summed_age = (
    region_cause_year_age.groupby(["region", "year", "cause"], as_index=False)["deaths"]
    .sum()
    .rename(columns={"deaths": "deaths_from_age"})
)

check_age = region_cause_year[["region", "year", "cause", "deaths"]].merge(
    summed_age, on=["region", "year", "cause"], how="outer"
)
check_age["diff"] = (check_age["deaths"] - check_age["deaths_from_age"]).abs()

print("Max difference:", check_age["diff"].max())
print("Rows with diff > 0.01:", (check_age["diff"] > 0.01).sum())

print()
print()
print()

AGE_ORDER = [
    "<28 days",
    "1-5 months",
    "6-11 months",
    "12-23 months",
    "2-4 years",
    "5-9 years",
    "10-14 years",
    "15-19 years",
    "20-24 years",
    "25-29 years",
    "30-34 years",
    "35-39 years",
    "40-44 years",
    "45-49 years",
    "50-54 years",
    "55-59 years",
    "60-64 years",
    "65-69 years",
    "70-74 years",
    "75-79 years",
    "80-84 years",
    "85-89 years",
    "90-94 years",
]

region_cause_year_age["age"] = pd.Categorical(
    region_cause_year_age["age"], categories=AGE_ORDER, ordered=True
)

print("Age categories set:", region_cause_year_age["age"].cat.categories.tolist()[:5], "...")

print()
print()
print()

cause_age = (
    region_cause_year_age.groupby(["cause", "age"], observed=True)["deaths"]
    .sum()
    .unstack(fill_value=0)
)

# Convert each row to percentages (so each cause sums to 100% across ages)
cause_age_pct = cause_age.div(cause_age.sum(axis=1), axis=0) * 100

print("Cause of death × age band, as % of that cause's total (all regions, 2000-2023):")
print(cause_age_pct.round(2).to_string())

print()
print()
print()

injuries_age = cause_age.loc["Injuries"]
injuries_age_pct = (injuries_age / injuries_age.sum() * 100).round(2)

print("Injuries: % of all injury deaths by age band (global, 2000-2023):")
print(injuries_age_pct.to_string())

print()
print()
print()

mn_age = cause_age.loc["Maternal and neonatal disorders"]
mn_age_pct = (mn_age / mn_age.sum() * 100).round(2)

print("Maternal + neonatal: % of all deaths from this cause, by age band:")
print(mn_age_pct.to_string())

print()
print()
print()

cancer_age = cause_age.loc["Total cancers"]
cancer_age_pct = (cancer_age / cancer_age.sum() * 100).round(2)

print("Total cancers: % of all cancer deaths by age band:")
print(cancer_age_pct.to_string())

print()
print()
print()

region_age = (
    region_cause_year_age.groupby(["region", "age"], observed=True)["deaths"]
    .sum()
    .unstack(fill_value=0)
)

region_age_pct = region_age.div(region_age.sum(axis=1), axis=0) * 100

# Show only the key age bands to keep the table readable
key_ages = [
    "<28 days", "1-5 months", "6-11 months", "12-23 months",
    "2-4 years", "5-9 years",
    "15-19 years", "25-29 years", "35-39 years",
    "50-54 years", "60-64 years", "70-74 years", "80-84 years",
]

print("Age distribution of deaths by region, % of that region's total (2000-2023):")
print(region_age_pct[key_ages].round(2).to_string())

print()
print()
print()

under5 = ["<28 days", "1-5 months", "6-11 months", "12-23 months", "2-4 years"]

region_age["under5_deaths"] = region_age[under5].sum(axis=1)
region_age["total_deaths"] = region_age[AGE_ORDER].sum(axis=1)
region_age["under5_share_%"] = (
    region_age["under5_deaths"] / region_age["total_deaths"] * 100
).round(2)

under5_table = region_age[["under5_share_%"]].sort_values("under5_share_%", ascending=False)

print("Share of all deaths occurring under 5 years of age, by region (2000-2023):")
print(under5_table.to_string())

print()
print()
print()

region_year_age = (
    region_cause_year_age.groupby(["region", "year", "age"], observed=True)["deaths"]
    .sum()
    .unstack(fill_value=0)
)

region_year_age["under5"] = region_year_age[under5].sum(axis=1)
region_year_age["total"] = region_year_age[AGE_ORDER].sum(axis=1)
region_year_age["under5_share"] = (
    region_year_age["under5"] / region_year_age["total"] * 100
)

# For each region, get 2000 and 2023
u5_2000 = region_year_age.xs(2000, level="year")["under5_share"].rename("u5_2000")
u5_2023 = region_year_age.xs(2023, level="year")["under5_share"].rename("u5_2023")

u5_change = pd.concat([u5_2000, u5_2023], axis=1)
u5_change["change_pp"] = (u5_change["u5_2023"] - u5_change["u5_2000"]).round(2)
u5_change = u5_change.sort_values("change_pp")

print("Change in under-5 share of deaths, 2000 → 2023 (percentage points):")
print(u5_change.round(2).to_string())

print()
print()
print()

ic_ages = region_cause_year_age[
    (region_cause_year_age["region"].isin(["India", "China"])) &
    (region_cause_year_age["year"] == 2023)
]

ic_age_profile = (
    ic_ages.groupby(["region", "age"], observed=True)["deaths"]
    .sum()
    .unstack(fill_value=0)
)

ic_age_pct = ic_age_profile.div(ic_age_profile.sum(axis=1), axis=0) * 100

print("Age distribution of deaths, India vs China, 2023 (%):")
print(ic_age_pct[key_ages].round(2).to_string())

print()
print()
print()

region_cause_year_age.to_csv("/content/sample_data/region_cause_year_age.csv", index=False)
cause_age_pct.to_csv("/content/sample_data/cause_by_age_global.csv")
region_age_pct.to_csv("/content/sample_data/age_by_region.csv")
print("Saved.")

Shape: (76032, 7)
        region  year               cause           age    deaths  deaths_upper  deaths_lower
0  Australasia  2000  Enteric infections    1-5 months  1.387891      1.738914      1.087703
1  Australasia  2000  Enteric infections   10-14 years  0.226666      0.285134      0.178033
2  Australasia  2000  Enteric infections  12-23 months  0.850639      1.068966      0.675077



Max difference: 4.656612873077393e-10
Rows with diff > 0.01: 0



Age categories set: ['<28 days', '1-5 months', '6-11 months', '12-23 months', '2-4 years'] ...



Cause of death × age band, as % of that cause's total (all regions, 2000-2023):
age                                           <28 days  1-5 months  6-11 months  12-23 months  2-4 years  5-9 years  10-14 years  15-19 years  20-24 years  25-29 years  30-34 years  35-39 years  40-44 years  45-49 years  50-54 years  55-59 years  60-64 years  65-69 years  70-74 years  75-79 years  80-84 years  85-89 years  90-94 years
cause                     

In [ ]:
#SECTION 8

uncertainty_region = (
    base.groupby("region")["uncertainty"]
    .agg(["mean", "median", "std", "count"])
    .round(3)
    .sort_values("mean", ascending=False)
)

print("Uncertainty by region (mean / median / std / n rows):")
print(uncertainty_region.to_string())

print()
print()
print()

uncertainty_cause = (
    base.groupby("cause")["uncertainty"]
    .agg(["mean", "median", "std"])
    .round(3)
    .sort_values("mean", ascending=False)
)

print("Uncertainty by cause:")
print(uncertainty_cause.to_string())

print()
print()
print()

uncertainty_age = (
    base.groupby("age")["uncertainty"]
    .agg(["mean", "median"])
    .round(3)
    .reindex(AGE_ORDER)
)

print("Uncertainty by age band (in chronological order):")
print(uncertainty_age.to_string())

print()
print()
print()

uncertainty_year = (
    base.groupby("year")["uncertainty"]
    .agg(["mean", "median"])
    .round(3)
)

print("Uncertainty by year:")
print(uncertainty_year.to_string())

print()
print()
print()

# Spearman correlation because the relationship is monotonic but not linear
corr_deaths_uncertainty = (
    base[["deaths", "uncertainty"]].dropna().corr(method="spearman")
)

print("Spearman correlation between deaths and uncertainty:")
print(corr_deaths_uncertainty.round(3).to_string())

print()
print()
print()

uncertainty_region_cause = (
    base.groupby(["region", "cause"])["uncertainty"]
    .mean()
    .round(3)
    .unstack()
)

# Sort regions by their overall mean uncertainty
region_order = uncertainty_region_cause.mean(axis=1).sort_values(ascending=False).index
uncertainty_region_cause = uncertainty_region_cause.loc[region_order]

print("Region × cause mean uncertainty:")
print(uncertainty_region_cause.to_string())

print()
print()
print()

top_uncertain = (
    base[["region", "year", "cause", "sex", "age", "deaths", "uncertainty"]]
    .dropna()
    .nlargest(20, "uncertainty")
    .round(3)
)

print("Top 20 most uncertain rows:")
print(top_uncertain.to_string(index=False))

print()
print()
print()

high_income = [
    "Australasia", "Western Europe", "North America",
    "Central Europe", "Eastern Europe",
]

low_middle_income = [
    "Central Asia", "Central Sub-Saharan Africa", "China", "India",
    "Latin America and Caribbean", "North Africa and Middle East",
    "Oceania", "Southeast Asia", "Southern Sub-Saharan Africa",
    "Western Sub-Saharan Africa", "Eastern Sub-Saharan Africa",
]

base["income_group"] = base["region"].apply(
    lambda r: "High income" if r in high_income else "Low/middle income"
)

uncertainty_group = (
    base.groupby("income_group")["uncertainty"]
    .agg(["mean", "median", "count"])
    .round(3)
)

print("Uncertainty by income group:")
print(uncertainty_group.to_string())

print()
print()
print()

focus_regions = ["India", "China", "Western Sub-Saharan Africa",
                 "Eastern Sub-Saharan Africa", "Southern Sub-Saharan Africa"]

focus_uncertainty = (
    base[base["region"].isin(focus_regions)]
    .groupby(["region", "cause"])["uncertainty"]
    .mean()
    .round(3)
    .unstack()
)

print("Mean uncertainty — India, China, and the sub-Saharan regions:")
print(focus_uncertainty.to_string())

print()
print()
print()


uncertainty_region.to_csv("/content/sample_data/uncertainty_by_region.csv")
uncertainty_cause.to_csv("/content/sample_data/uncertainty_by_cause.csv")
uncertainty_region_cause.to_csv("/content/sample_data/uncertainty_region_cause.csv")
base.to_csv("/content/sample_data/base_with_income_group.csv", index=False)
print("Saved.")

Uncertainty by region (mean / median / std / n rows):
                               mean  median    std  count
region                                                   
Oceania                       1.051   1.014  0.457   9240
Central Sub-Saharan Africa    1.024   0.997  0.373   9240
Southeast Asia                0.784   0.687  0.422   9240
Western Sub-Saharan Africa    0.783   0.720  0.343   9240
North Africa and Middle East  0.765   0.687  0.396   9240
India                         0.757   0.642  0.384   9240
Eastern Sub-Saharan Africa    0.703   0.611  0.352   9246
China                         0.675   0.582  0.327   9240
Southern Sub-Saharan Africa   0.670   0.537  0.409   9240
Australasia                   0.560   0.370  0.535   9240
Central Asia                  0.461   0.323  0.380   9240
Central Europe                0.401   0.246  0.416   9240
Eastern Europe                0.389   0.282  0.360   9240
North America                 0.352   0.314  0.200   9240
Western Europe    

In [ ]:
#SECTION 9

# 1) leading cause per region in 2023
latest = region_cause_year[region_cause_year["year"] == 2023]
pivot_2023 = latest.pivot_table(index="region", columns="cause",
                                values="share", aggfunc="sum")
leading_cause = pivot_2023.idxmax(axis=1).rename("leading_cause_2023")
leading_share = pivot_2023.max(axis=1).round(2).rename("leading_share_2023")

# 2) under-5 share in 2023
u5_2023 = (
    region_year_age.xs(2023, level="year")["under5_share"].round(2)
    .rename("under5_share_2023")
)

# 3) transition ratio in 2023
ratio_2023 = category_totals.xs(2023, level="year")["ratio"].rename("transition_ratio_2023")

# 4) injuries male ratio (all time)
inj_ratio = injuries_pivot["ratio_M_F"].rename("injuries_male_ratio")

# 5) HIV male ratio (all time)
hiv_ratio = hiv_pivot["ratio_M_F"].rename("hiv_male_ratio")

# 6) mean uncertainty (all time)
unc_mean = uncertainty_region["mean"].rename("mean_uncertainty")

# 7) total deaths (all time)
total_deaths = (
    region_year_total.groupby("region")["total_deaths"].sum()
    .round(0).rename("total_deaths")
)

# 8) income group
income_map = base.groupby("region")["income_group"].first()

# combine all into one scorecard
scorecard = pd.concat([
    total_deaths, leading_cause, leading_share, u5_2023,
    ratio_2023, inj_ratio, hiv_ratio, unc_mean, income_map
], axis=1)

scorecard = scorecard[[
    "income_group", "total_deaths",
    "leading_cause_2023", "leading_share_2023",
    "under5_share_2023", "transition_ratio_2023",
    "injuries_male_ratio", "hiv_male_ratio",
    "mean_uncertainty",
]]

print("Region scorecard built:", scorecard.shape)
print(scorecard.head(3).to_string())

print()
print()
print()

print("=" * 100)
print("REGION SCORECARD — key metrics by world region")
print("=" * 100)
print(scorecard.to_string())

print()
print()
print()

total_global_deaths = base["deaths"].sum()
total_rows = base.shape[0]
n_regions = base["region"].nunique()
n_causes = base["cause"].nunique()

# global cause shares (all years)
global_cause = (
    region_cause_year.groupby("cause")["deaths"].sum()
    .sort_values(ascending=False)
)
global_cause_share = (global_cause / global_cause.sum() * 100).round(2)

print("GLOBAL SUMMARY — IHME GBD 2023 Data, 2000-2023")
print("=" * 60)
print(f"Regions analysed     : {n_regions}")
print(f"Causes analysed      : {n_causes}")
print(f"Rows in base table   : {total_rows:,}")
print(f"Total deaths covered : {total_global_deaths:,.0f}")
print()
print("Global cause share (% of all deaths in dataset):")
print(global_cause_share.to_string())

print()
print()
print()

uncertainty_recent = (
    base[base["year"].between(2015, 2023)]
    .groupby(["region", "year"])["uncertainty"]
    .mean()
    .unstack()
    .round(3)
)

uncertainty_recent["change_2019_2023"] = (
    uncertainty_recent[2023] - uncertainty_recent[2019]
).round(3)

uncertainty_recent = uncertainty_recent.sort_values(
    "change_2019_2023", ascending=False
)

print("Mean uncertainty by region, 2015-2023, and change from 2019 to 2023:")
print(uncertainty_recent.to_string())

print()
print()
print()


uncertainty_cause_year = (
    base[base["year"].between(2018, 2023)]
    .groupby(["cause", "year"])["uncertainty"]
    .mean()
    .unstack()
    .round(3)
)

uncertainty_cause_year["change_2019_2023"] = (
    uncertainty_cause_year[2023] - uncertainty_cause_year[2019]
).round(3)

uncertainty_cause_year = uncertainty_cause_year.sort_values(
    "change_2019_2023", ascending=False
)

print("Mean uncertainty by cause, 2018-2023, and change from 2019 to 2023:")
print(uncertainty_cause_year.to_string())

print()
print()
print()

# The big ones
scorecard.to_csv("/content/sample_data/FINAL_region_scorecard.csv")
global_cause_share.to_csv("/content/sample_data/FINAL_global_cause_shares.csv")
pivot_2023.to_csv("/content/sample_data/FINAL_cause_share_2023.csv")

# Supporting tables
trend.to_csv("/content/sample_data/FINAL_cause_trend_2000_2023.csv", index=False)
under5_table.to_csv("/content/sample_data/FINAL_under5_share.csv")
uncertainty_region.to_csv("/content/sample_data/FINAL_uncertainty_by_region.csv")
uncertainty_cause.to_csv("/content/sample_data/FINAL_uncertainty_by_cause.csv")

print("Exported 7 FINAL_*.csv files.")

print()
print()
print()

print("=" * 80)
print("TOP FINDINGS — IHME GBD 2023, 16 world regions, 2000-2023")
print("=" * 80)
print()
print("1. TOTAL CANCERS now lead in 11 of 16 regions, up from roughly")
print("   a third of deaths in 2000 to over 42% globally by 2023.")
print()
print("2. HIV/AIDS collapsed in Southern Sub-Saharan Africa:")
print("   -22.69 percentage points in 23 years — the largest single")
print("   cause decline anywhere in the dataset.")
print()
print("3. INJURIES are 2.31x more common in men worldwide, but the")
print("   ratio ranges from 1.68 (Western Europe) to 3.92 (Latin America).")
print()
print("4. HIV is FEMALE-dominant in every Sub-Saharan region (0.64-0.92)")
print("   but MALE-dominant in every other region (2.5-2.9). Same disease,")
print("   opposite demographic signature.")
print()
print("5. UNDER-5 DEATHS are 53.68% of all deaths in Western Sub-Saharan")
print("   Africa and 0.71% in Western Europe — a 75x gap.")
print()
print("6. INDIA vs CHINA: newborns make up 9.33% of India's deaths but")
print("   only 0.29% of China's. Same population size, radically different")
print("   child survival.")
print()
print("7. COVID hit in staggered waves: Eastern Europe peaked in 2021,")
print("   Australasia in 2022, China in 2023. Not one pandemic.")
print()
print("8. UNCERTAINTY is 1.77x higher in low/middle-income regions than")
print("   high-income (0.725 vs 0.410).")
print()
print("9. UNCERTAINTY rose sharply after 2019, peaking at 0.722 in 2023 —")
print("   the worst data-confidence year in 24 years.")
print()
print("10. NTD/malaria is the most uncertain cause globally (1.092), but")
print("    its uncertainty is high even in rich regions because the true")
print("    value is close to zero there.")
print("=" * 80)

Region scorecard built: (16, 9)
                     income_group  total_deaths leading_cause_2023  leading_share_2023  under5_share_2023  transition_ratio_2023  injuries_male_ratio  hiv_male_ratio  mean_uncertainty
region                                                                                                                                                                                 
Australasia           High income     1681217.0      Total cancers               77.45               0.93                  13.23                 1.92            2.93             0.560
Central Asia    Low/middle income     4538512.0      Total cancers               42.29              24.44                   1.73                 3.13            1.87             0.461
Central Europe        High income    11124039.0      Total cancers               70.84               0.64                   5.35                 2.70            2.08             0.401



REGION SCORECARD — key metrics by world regio